# Cost comparison: three ways to run the questionnaire responder
Run 1 answered all 60 Globex questions on Claude Opus for an estimated ~$2.20. This notebook runs the **same questionnaire** through two cheaper pipelines and scores all three against the same answer key.

| Step | **A. Baseline** (`demo.ipynb`) | **B. Claude, tiered** | **C. Jev hybrid** |
|---|---|---|---|
| Classify | Haiku | Haiku | **Jev**: specialist choice + 4 yes/no commitment checks |
| Find evidence | Opus agent searches | Haiku agent searches | Code search (FTS) + **Jev** relevance gate per section |
| Write the answer | Opus | Sonnet | Sonnet |
| Check the answer | Opus | Haiku | Haiku extracts claims + quotes → code checks quotes and numbers → **Jev** judges supports / contradicts / says nothing |

**Jev** (TypeSafe AI) is a *decision model*. It doesn't write text: it answers typed yes/no and pick-one questions with **probabilities**. Pipeline C gives it every judgement call and leaves Claude only the writing, following two of TypeSafe's published recipes ("classifying RAG passages" and "double-checking citations").

**How to use it**
1. Add `ANTHROPIC_API_KEY` and `TYPESAFE_API_KEY` in Colab Secrets.
2. Pick `PIPELINE` and `RUN_MODE` in section 1. Start with `RUN_MODE = "smoke"` (10 questions, a few cents) to catch problems cheaply.
3. Run top to bottom. The **pre-flight check** in section 9 must say 🟢 before the paid run in section 10.
4. Section 14 builds the A / B / C scorecard from every audit file in the session.

Costs are **measured** from the token counts each API returns, not estimated.

## 1. Setup and settings

In [ ]:
!pip install -q langgraph langchain-anthropic typesafe-sdk

In [ ]:
import os
from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
os.environ["TYPESAFE_API_KEY"] = userdata.get("TYPESAFE_API_KEY")

PIPELINE = "C"         # "B" = Claude tiered (Haiku + Sonnet), "C" = Jev hybrid (Jev + Haiku + Sonnet)
RUN_MODE = "smoke"     # "smoke" = 10 hand-picked questions, "full" = the whole questionnaire

HAIKU, SONNET = "claude-haiku-5-5", "claude-sonnet-5-5"
DRAFT_MODEL = SONNET   # the only model that writes customer-facing text
JEV_MODEL = "jev-latest"

# USD per million tokens (input, output). Claude: Anthropic list prices, Nov 2026 (Haiku: prompts
# under 100k tokens). Jev: published price, output currently free; check your TypeSafe console.
PRICES = {"claude-opus-5-5": (4.00, 20.00), SONNET: (2.00, 10.00), HAIKU: (0.10, 0.50), "jev": (0.042, 0.0)}

# Same confidence rules as the baseline. Placeholders until the Phase 4 evals calibrate them.
THRESHOLDS = {"high_min_support": 1.0, "high_min_explicit": 1.0, "medium_min_support": 0.75,
              "min_retrieval_score": 4.0}
# Jev starting points, from TypeSafe's docs (Noul page, citation and RAG-passage cookbooks).
JEV_T = {"yes": 0.7,             # probability above this counts as "yes"
         "no": 0.4,              # below this counts as "no"; in between = uncertain, goes to a human
         "gate_relevant": 0.45,  # a section must at least be on topic...
         "gate_evidence": 0.55,  # ...and contain usable evidence to reach the writer
         "complete": 0.6,        # sections together answer every part of the question
         "support_conf": 0.8}    # confidence needed to accept "supports" without a human

AUTO_APPROVE_LEVELS = {"High"}
ESCALATE_TEXT = "Not covered by policy, escalate"
SEARCH_K = 6

# Smoke test: 10 questions covering every expected outcome, including the ones that tripped run 2.
SMOKE_REFS = {"1.1.1", "AI-02", "3.5", "1.5.3",      # answerable -> should auto-approve
              "1.3.2", "1.5.1",                      # implied-only / partly covered -> human review
              "1.5.4", "AI-20",                      # restricted -> human review
              "1.4.2", "AI-04"}                      # traps -> escalate
print(f"Pipeline {PIPELINE}, run mode '{RUN_MODE}'.")

## 2. Load and index the Acme SaaS policy pack
Same as `demo.ipynb`. Upload `acme-policy-pack.zip` when asked.

In [ ]:
import glob, pathlib, zipfile

if not glob.glob("**/data/policies/*.md", recursive=True):
    from google.colab import files
    uploaded = files.upload()                      # choose acme-policy-pack.zip
    for name in uploaded:
        if name.endswith(".zip"):
            zipfile.ZipFile(name).extractall(".")

POLICY_DIR = pathlib.Path(sorted(glob.glob("**/data/policies", recursive=True))[0])
policy_files = sorted(POLICY_DIR.glob("*.md"))
print(f"{len(policy_files)} policy docs in {POLICY_DIR}:")
for p in policy_files:
    print("  -", p.name)

In [ ]:
import re

def parse_doc(path):
    text = path.read_text(encoding="utf-8")
    meta = {}
    m = re.match(r"^---\n(.*?)\n---\n", text, re.S)          # front matter
    if m:
        for line in m.group(1).splitlines():
            key, _, value = line.partition(":")
            meta[key.strip()] = value.strip()
        text = text[m.end():]
    chunks = []
    for section in re.split(r"\n(?=## )", text):
        head = re.match(r"## ([A-Z]+-\d+) (.+)", section)
        if not head:
            continue
        chunks.append({
            "section_id": head.group(1),
            "heading": head.group(2).strip(),
            "doc_id": meta.get("doc_id", ""),
            "doc_title": meta.get("title", path.stem),
            "version": meta.get("version", ""),
            "text": section[head.end():].strip(),
        })
    return chunks

chunks = [c for p in policy_files for c in parse_doc(p)]
print(f"{len(chunks)} sections. Example:\n")
print(chunks[0]["section_id"], "|", chunks[0]["doc_title"], "|", chunks[0]["heading"])
print(chunks[0]["text"])

In [ ]:
import sqlite3

DB_PATH = "acme_policies.db"
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH, check_same_thread=False)
conn.row_factory = sqlite3.Row
conn.execute('''
    CREATE VIRTUAL TABLE passages USING fts5(
        section_id UNINDEXED, doc_id UNINDEXED, doc_title UNINDEXED, version UNINDEXED,
        heading, text,
        tokenize = 'porter unicode61'
    )''')
conn.executemany(
    "INSERT INTO passages VALUES (:section_id, :doc_id, :doc_title, :version, :heading, :text)",
    chunks)
conn.commit()

STOPWORDS = set('''a an and are as at be by can do does for from has have how i if in
into is it its of on or our please provide the their there this to us we what when
where which who will with you your yours any all describe detail details'''.split())

def to_fts_query(text):
    terms = [t for t in re.findall(r"[a-z0-9]+", text.lower())
             if t not in STOPWORDS and len(t) > 1]
    return " OR ".join(f'"{t}"' for t in dict.fromkeys(terms))   # quoted = safe from FTS syntax

def search(query, k=4):
    fts = to_fts_query(query)
    if not fts:
        return []
    rows = conn.execute('''
        SELECT section_id, doc_title, heading, text,
               -bm25(passages, 0, 0, 0, 0, 2.0, 1.0) AS score
        FROM passages WHERE passages MATCH ?
        ORDER BY score DESC LIMIT ?''', (fts, k)).fetchall()
    return [dict(r) for r in rows]

def format_hits(hits):
    return "\n\n".join(
        f"[{h['section_id']}] {h['doc_title']}: {h['heading']} (score {h['score']:.2f})\n{h['text']}"
        for h in hits)

print(f"Indexed {conn.execute('SELECT count(*) FROM passages').fetchone()[0]} sections.\n")
for q in ["Do you train AI models on customer data?", "Is data encrypted at rest?", "bug bounty programme"]:
    print(f"Q: {q}")
    for h in search(q, k=3):
        print(f"   {h['score']:5.2f}  {h['section_id']:7} {h['heading']}")
    print()

# The batch run calls search from several threads at once, so guard the SQLite connection.
import threading
_search_lock = threading.Lock()
if "_search_unlocked" not in globals():
    _search_unlocked = search
def search(query, k=4):
    with _search_lock:
        return _search_unlocked(query, k)

## 3. Measure what every call costs
Every Claude and Jev response reports its token counts. `track()` adds them up per model, so section 11 can show the **actual** cost of the run.

In [ ]:
import threading
import pandas as pd
from collections import defaultdict

_usage_lock = threading.Lock()
USAGE = defaultdict(lambda: {"calls": 0, "input_tokens": 0, "output_tokens": 0})

def track(model, input_tokens, output_tokens):
    with _usage_lock:
        u = USAGE[model]
        u["calls"] += 1
        u["input_tokens"] += input_tokens or 0
        u["output_tokens"] += output_tokens or 0

def track_msg(model, msg):
    um = getattr(msg, "usage_metadata", None) or {}
    track(model, um.get("input_tokens", 0), um.get("output_tokens", 0))

def cost_table(usage=None):
    rows = []
    for model, u in (usage or USAGE).items():
        p_in, p_out = PRICES.get(model, (0, 0))
        rows.append({"model": model, **u,
                     "cost_usd": u["input_tokens"] / 1e6 * p_in + u["output_tokens"] / 1e6 * p_out})
    df = pd.DataFrame(rows, columns=["model", "calls", "input_tokens", "output_tokens", "cost_usd"])
    return df.sort_values("cost_usd", ascending=False).reset_index(drop=True)

print("Usage tracking ready.")

## 4. Shared schemas, prompts and checks
The classifier definition, drafting prompt and grounding prompt include the three fixes from run 2.

In [ ]:
from typing import Literal, TypedDict
from pydantic import BaseModel, Field

class Classification(BaseModel):
    category: Literal["security", "ai_governance", "privacy"]
    asks_for_commitment: bool = Field(description=(
        "True ONLY if the question asks the vendor to go beyond describing its current practice: "
        "to guarantee or warrant an outcome, agree new or different contractual terms, accept "
        "liability or indemnity, agree SLA penalties or service credits, or hand over a confidential "
        "report. False for questions asking what the vendor's existing policy, timeframe, notice "
        "period or process is, even when the answer describes something the vendor already does."))
    reason: str = Field(description="One short sentence explaining the category.")

class DraftAnswer(BaseModel):
    covered: bool = Field(description="False if the passages do not answer the question.")
    answer: str = Field(description="The answer as the vendor would write it. Empty if not covered.")
    cited_section_ids: list[str] = Field(description="Section IDs the answer relies on, e.g. ['AIG-3'].")

class Claim(BaseModel):
    claim: str = Field(description="One factual statement taken from the answer.")
    cited_section_id: str = Field(description="The one section that supports it, or '' if none.")
    evidence_quote: str = Field(description="Exact words copied from that section that support the claim, or ''.")
    verdict: Literal["explicit", "inferred", "unsupported"] = Field(description=(
        "explicit: the section states it directly; inferred: it follows from the section but is not "
        "stated; unsupported: no cited section supports it."))

class GroundingReport(BaseModel):
    claims: list[Claim]

class ExtractedClaim(BaseModel):
    claim: str = Field(description="One factual statement taken from the answer.")
    cited_section_id: str = Field(description="The one section that supports it, or '' if none.")
    evidence_quote: str = Field(description="Exact words copied from that section that support the claim, or ''.")

class ClaimList(BaseModel):
    claims: list[ExtractedClaim]

class QState(TypedDict, total=False):
    question: str
    classification: dict
    restricted: bool
    restricted_reasons: list[str]
    caps: list[str]       # reasons that cap confidence at Medium (pipeline C: uncertain Jev answers)
    jev: dict             # raw Jev probabilities, logged to the audit trail
    search_log: list[str]
    passages: dict        # section_id -> passage dict
    draft: dict
    grounding: dict
    result: dict

print("Schemas ready.")

In [ ]:
import re
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool

LLM = {m: ChatAnthropic(model=m, max_tokens=2048, max_retries=3) for m in (HAIKU, SONNET)}

def structured(model, schema, messages):
    '''Structured call that also records token usage.'''
    out = LLM[model].with_structured_output(schema, method="json_schema", include_raw=True).invoke(messages)
    track_msg(model, out["raw"])
    if out.get("parsing_error"):
        raise out["parsing_error"]
    return out["parsed"]
CLASSIFIER_PROMPT = '''You route questions from a customer security / AI due-diligence questionnaire
to the right specialist at a SaaS vendor.
- security: infosec programme, certifications, access control, encryption, vulnerability management,
  hosting, network, identity/SSO, incident response, business continuity.
- ai_governance: AI features, models, model providers, training on data, AI risk management,
  AI regulation (EU AI Act, NIST AI RMF, ISO 42001), AI testing and oversight.
- privacy: personal data, GDPR, DPAs, data retention and deletion, sub-processors, data transfers,
  data subject rights.'''

COMMON_RULES = '''You work for Acme SaaS (the vendor) and answer questions from a prospect's
due-diligence questionnaire using ONLY Acme's internal policy pack.
Rules:
- Every statement must come from a retrieved policy section. Never use outside knowledge about
  Acme or assume typical industry practice.
- If the sections do not answer the question, say so. Do not guess or fill gaps.
- Do not add guarantees, promises or commitments that the policy does not state.'''

SPECIALIST_FOCUS = {
    "security": "You are Acme's security specialist. Be precise about certifications, scopes, "
                "time frames and technical controls.",
    "ai_governance": "You are Acme's AI governance specialist. Be precise about how customer data "
                     "is (and is not) used by AI, model providers, human oversight and regulatory "
                     "positioning. Never overstate certifications that are only in progress.",
    "privacy": "You are Acme's privacy specialist. Be precise about controller/processor roles, "
               "retention periods, deletion, sub-processors and data locations.",
}

DRAFT_PROMPT = '''Write the questionnaire answer from the passages provided.
- Answer exactly what was asked, in 1 to 3 sentences of plain professional English, in Acme's voice ("Acme ...", "We ...").
- Include the specifics the question asks for (numbers, time frames, standards). Do not add related
  facts the question didn't ask about: every extra sentence is one more claim a reviewer must check.
- For a yes/no question, start with "Yes." or "No." only when a passage states it directly.
- Do not put section IDs in the answer text; list them in cited_section_ids.
- If the passages do not answer the question, set covered=false, answer="" and cited_section_ids=[].'''

GROUNDING_PROMPT = '''You are an auditor checking a questionnaire answer against its cited policy sections.
Split the answer into its individual factual claims. A bare "Yes." or "No." is not a separate claim:
treat it as part of the claim that follows. For each claim, pick the one cited section that supports
it and copy the exact supporting words from that section into evidence_quote (verbatim, no
paraphrasing). Verdicts:
- explicit: the quoted words state the same fact, even if the answer words it differently;
- inferred: the claim adds meaning the quote does not contain (a conclusion, an absence, a wider scope);
- unsupported: no cited section supports it.
Be strict about meaning, not wording: anything not in the cited sections is unsupported.'''
print("Models and prompts ready.")

EXTRACT_PROMPT = '''Split the questionnaire answer into its individual factual claims. A bare "Yes." or
"No." is not a separate claim: treat it as part of the claim that follows. For each claim, pick the one
cited section that supports it and copy the exact supporting words from that section into evidence_quote:
verbatim, no paraphrasing, no ellipses. If no cited section supports a claim, leave both fields empty.'''

RESTRICTED_RULES = {
    "guarantee":           r"\bguarantee",
    "warranty":            r"\bwarrant",
    "liability":           r"\bliabilit|\bliable\b",
    "indemnity":           r"\bindemn",
    "commitment":          r"\bcommit(ment)? to\b|\bcontractual|\bin (the|our|your) contract\b",
    "sla / credits":       r"\bSLA\b|service credit|\bpenalt",
    "confidential report": r"\b(full|complete) (pen(etration)? test|audit|soc ?2) report",
}
def keyword_flags(q):
    return [f"keyword rule: {name}" for name, rx in RESTRICTED_RULES.items() if re.search(rx, q, re.I)]

def _norm(s):
    return re.sub(r"\s+", " ", re.sub(r"[^\w\s%.]", " ", s.lower())).strip()

NUM = re.compile(r"\d+(?:\.\d+)?")
def numbers_match(claim, section_text):
    '''Every number in the claim must appear in the cited section (Jev is weak at numbers).'''
    have = set(NUM.findall(section_text))
    return all(n in have for n in NUM.findall(claim))
print("Prompts and checks ready.")

## 5. Shared confidence scoring
Both pipelines produce claims with the same verdicts (explicit / inferred / unsupported), so they share one scoring function. Pipeline C can also add **caps**: reasons that hold an answer at Medium, such as Jev being unsure whether a question asks for a commitment.

In [ ]:
def score(state: QState) -> dict:
    d, claims = state["draft"], state["grounding"]["claims"]
    n = len(claims)
    support = sum(c["verdict"] != "unsupported" for c in claims) / n if n else 0.0
    explicit = sum(c["verdict"] == "explicit" for c in claims) / n if n else 0.0
    best = max((state["passages"][s]["score"] for s in d["cited_section_ids"] if s in state["passages"]), default=0.0)
    caps = state.get("caps", [])
    signals = {"library_match": False, "best_retrieval_score": round(best, 2), "claims": n,
               "claim_support": round(support, 2), "explicit_evidence": round(explicit, 2),
               "restricted": state["restricted"], **state.get("jev", {})}
    reasons, answer = [], d["answer"]

    if not d["covered"] or n == 0 or support == 0:
        level, answer = "Low", ESCALATE_TEXT
        reasons.append("policy pack doesn't cover this question")
    elif (support >= THRESHOLDS["high_min_support"] and explicit >= THRESHOLDS["high_min_explicit"]
          and (PIPELINE == "C" or best >= THRESHOLDS["min_retrieval_score"]) and not caps):
        level = "High"
        reasons.append("every claim is backed by a verified quote from a cited section")
    elif support >= THRESHOLDS["medium_min_support"]:
        level = "Medium"
        if explicit < THRESHOLDS["high_min_explicit"]:
            reasons.append("some claims are inferred rather than stated")
        if support < 1:
            reasons.append("some claims are unsupported")
        reasons += caps
    else:
        level = "Low"
        reasons.append(f"only {support:.0%} of claims are supported")
    if any(c.get("contradicted") for c in claims):
        level = "Low"
        reasons.append("a claim contradicts the cited policy")
    if state["restricted"]:
        level = "Low"
        reasons += [f"restricted topic, {r}" for r in state["restricted_reasons"]]

    status = ("escalate" if answer == ESCALATE_TEXT
              else "auto_approve" if level in AUTO_APPROVE_LEVELS else "human_review")
    return {"result": {"question": state["question"], "category": state["classification"]["category"],
                       "answer": answer, "draft_answer": d["answer"],
                       "sources": d["cited_section_ids"] if answer != ESCALATE_TEXT else [],
                       "confidence": level, "status": status, "reasons": reasons, "signals": signals}}
print("Scoring ready.")

## 6. Pipeline B: Claude, tiered
Same design as the baseline, with cheaper models: **Haiku** classifies, searches and checks; **Sonnet** writes.

In [ ]:
MAX_TOOL_ROUNDS = 4

def classify_b(state: QState) -> dict:
    q = state["question"]
    c = structured(HAIKU, Classification, [SystemMessage(CLASSIFIER_PROMPT), HumanMessage(q)])
    reasons = keyword_flags(q)
    if c.asks_for_commitment:
        reasons.append(f"classifier: asks for a commitment ({c.reason})")
    return {"classification": c.model_dump(), "restricted": bool(reasons), "restricted_reasons": reasons}

def specialist_b(state: QState) -> dict:
    q, category = state["question"], state["classification"]["category"]
    found, log = {}, []
    def remember(hits):
        for h in hits:
            if h["section_id"] not in found or h["score"] > found[h["section_id"]]["score"]:
                found[h["section_id"]] = h

    @tool
    def search_policies(query: str) -> str:
        '''Search Acme SaaS's internal policy pack. Use short keyword queries
        (e.g. "encryption at rest keys"). Returns matching sections with their IDs.'''
        hits = search(query)
        log.append(query)
        remember(hits)
        return format_hits(hits) or "No matching sections."

    remember(search(q))
    log.append(f"(question) {q}")
    agent = LLM[HAIKU].bind_tools([search_policies])
    messages = [SystemMessage(COMMON_RULES + "\n\n" + SPECIALIST_FOCUS[category]),
                HumanMessage(f"Questionnaire question: {q}\n\nSearch the policy pack with 1-3 different "
                             "keyword queries to find every relevant section. When you have enough, reply DONE.")]
    for _ in range(MAX_TOOL_ROUNDS):
        ai = agent.invoke(messages)
        track_msg(HAIKU, ai)
        messages.append(ai)
        if not ai.tool_calls:
            break
        for tc in ai.tool_calls:
            messages.append(ToolMessage(search_policies.invoke(tc["args"]), tool_call_id=tc["id"]))

    if not found:
        return {"passages": {}, "search_log": log, "draft": {"covered": False, "answer": "", "cited_section_ids": []}}
    ranked = sorted(found.values(), key=lambda h: -h["score"])
    draft = structured(DRAFT_MODEL, DraftAnswer, [
        SystemMessage(COMMON_RULES + "\n\n" + SPECIALIST_FOCUS[category] + "\n\n" + DRAFT_PROMPT),
        HumanMessage(f"Question: {q}\n\nPolicy sections:\n\n{format_hits(ranked)}")])
    d = draft.model_dump()
    d["cited_section_ids"] = [s for s in dict.fromkeys(d["cited_section_ids"]) if s in found]
    return {"passages": found, "search_log": log, "draft": d}

def grounding_b(state: QState) -> dict:
    d = state["draft"]
    if not d["covered"] or not d["cited_section_ids"]:
        return {"grounding": {"claims": []}}
    cited = {s: state["passages"][s] for s in d["cited_section_ids"]}
    report = structured(HAIKU, GroundingReport, [
        SystemMessage(GROUNDING_PROMPT),
        HumanMessage(f"Question: {state['question']}\n\nAnswer: {d['answer']}\n\n"
                     f"Cited sections:\n\n{format_hits(list(cited.values()))}")])
    claims = []
    for c in report.claims:
        c = c.model_dump()
        section = cited.get(c["cited_section_id"])
        c["quote_found"] = bool(section and c["evidence_quote"].strip()
                                and _norm(c["evidence_quote"]) in _norm(section["text"]))
        if c["verdict"] != "unsupported" and section is None:
            c["verdict"], c["note"] = "unsupported", "cited a section that wasn't retrieved"
        elif c["verdict"] == "explicit" and not c["quote_found"]:
            c["verdict"], c["note"] = "inferred", "quote not found verbatim; downgraded"
        claims.append(c)
    return {"grounding": {"claims": claims}}

print("Pipeline B nodes defined.")

## 7. Pipeline C: Jev hybrid
- **Classify (Jev, 1 request).** A *Choice* picks the specialist. Four separate *Noul* (yes/no) questions check for a guarantee, contract terms, liability and a confidential document. TypeSafe's docs say to split judgements like this rather than ask one broad question. A probability above 0.7 counts as **restricted**. Between 0.3 and 0.7 is **uncertain**, which caps the answer at Medium so a human looks at it. The keyword rule stays as a backstop.
- **Find evidence (code + Jev).** One plain search pulls the top 6 sections. For each section, Jev answers *"is it on topic?"* and *"does it contain usable evidence?"*. Only sections passing both reach the writer. If none pass, the question escalates **without any Claude call**. One more Jev request asks whether the surviving sections answer *every part* of the question.
- **Write (Sonnet).** From the sections that passed the gate only.
- **Check.** Haiku lists each claim with a cited section and a verbatim quote. Code then confirms the quote exists, and that every number in the claim appears in the section, since TypeSafe lists numbers and dates as Jev's weak spot. Finally Jev judges whether the section **supports**, **contradicts** or **says nothing about** the claim.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from typesafe_sdk import TypeSafeClient, Noul, Choice

jev_client = TypeSafeClient(model=JEV_MODEL)          # reads TYPESAFE_API_KEY; retries 429s/timeouts itself
_jev_pool = ThreadPoolExecutor(max_workers=8)

def ask_jev(state, questions):
    r = jev_client.system_one(state=state, questions=questions, model=JEV_MODEL)
    track("jev", r.usage.input_tokens, r.usage.output_tokens)
    return r.answers

CLASSIFY_QS = {
    "category": Choice(
        instructions="Which specialist at a SaaS vendor should answer this due-diligence question?",
        criteria={
            "security": "Information security programme, certifications, access control, encryption, "
                        "vulnerabilities, hosting, network, identity/SSO, incident response, continuity.",
            "ai_governance": "AI features, models and model providers, training on data, AI risk "
                             "management, AI regulation (EU AI Act, NIST AI RMF, ISO 42001), AI testing.",
            "privacy": "Personal data, GDPR, data processing agreements, retention and deletion, "
                       "sub-processors, data transfers, data subject rights."}),
    "asks_guarantee": Noul(instructions="The question asks the vendor to guarantee or warrant an outcome."),
    "asks_contract_terms": Noul(instructions=(
        "The question asks the vendor to agree new contractual terms, an SLA, penalties or service "
        "credits, rather than to describe its existing policy or practice.")),
    "asks_liability": Noul(instructions="The question asks the vendor to accept liability or to indemnify the customer."),
    "asks_confidential_document": Noul(instructions=(
        "The question asks the vendor to hand over a complete confidential document, such as a full "
        "audit report or penetration test report.")),
}
COMMIT_KEYS = ["asks_guarantee", "asks_contract_terms", "asks_liability", "asks_confidential_document"]

def classify_c(state: QState) -> dict:
    q = state["question"]
    a = ask_jev({"questionnaire_question": q}, CLASSIFY_QS)
    p = {k: round(a[k].noul, 3) for k in COMMIT_KEYS}
    reasons = keyword_flags(q) + [f"Jev: {k} p={v:.2f}" for k, v in p.items() if v >= JEV_T["yes"]]
    caps = [f"Jev unsure whether it {k.replace('_', ' ')} (p={v:.2f})"
            for k, v in p.items() if JEV_T["no"] <= v < JEV_T["yes"]]
    cat = a["category"]
    return {"classification": {"category": cat.choice, "asks_for_commitment": bool(reasons),
                               "reason": f"Jev confidence {cat.confidence:.2f}"},
            "restricted": bool(reasons), "restricted_reasons": reasons, "caps": caps,
            "jev": {"category_conf": round(cat.confidence, 3), **{f"p_{k}": v for k, v in p.items()}}}

GATE_QS = {
    "is_relevant": Noul(instructions="The passage addresses the subject of the question."),
    "has_evidence": Noul(instructions="The passage states information that directly answers the question, or at least one part of it."),
}
COMPLETE_Q = {"answers_every_part": Noul(instructions=(
    "Taken together, the passages contain the information needed to answer every part of the question."))}

def retrieve_gate_c(state: QState) -> dict:
    q = state["question"]
    hits = search(q, k=SEARCH_K)
    def judge(h):
        a = ask_jev({"question": q, "passage": {"id": h["section_id"], "title": h["heading"], "text": h["text"]}}, GATE_QS)
        return h, a["is_relevant"].noul, a["has_evidence"].noul
    judged = list(_jev_pool.map(judge, hits))
    kept = {h["section_id"]: {**h, "p_relevant": round(r, 3), "p_evidence": round(e, 3)}
            for h, r, e in judged if r >= JEV_T["gate_relevant"] and e >= JEV_T["gate_evidence"]}
    jev = {**state.get("jev", {}), "gate": {h["section_id"]: [round(r, 2), round(e, 2)] for h, r, e in judged},
           "sections_kept": len(kept)}
    caps = list(state.get("caps", []))
    if kept:
        a = ask_jev({"question": q, "passages": [{"id": s, "text": h["text"]} for s, h in kept.items()]}, COMPLETE_Q)
        jev["p_complete"] = round(a["answers_every_part"].noul, 3)
        if jev["p_complete"] < JEV_T["complete"]:
            caps.append(f"sections may not answer every part of the question (p={jev['p_complete']:.2f})")
    return {"passages": kept, "jev": jev, "caps": caps,
            "search_log": [f"(question) {q}: {len(hits)} found, {len(kept)} passed the Jev gate"]}

def draft_c(state: QState) -> dict:
    q, category = state["question"], state["classification"]["category"]
    if not state["passages"]:
        return {"draft": {"covered": False, "answer": "", "cited_section_ids": []}}
    ranked = sorted(state["passages"].values(), key=lambda h: -h["score"])
    d = structured(DRAFT_MODEL, DraftAnswer, [
        SystemMessage(COMMON_RULES + "\n\n" + SPECIALIST_FOCUS[category] + "\n\n" + DRAFT_PROMPT),
        HumanMessage(f"Question: {q}\n\nPolicy sections:\n\n{format_hits(ranked)}")]).model_dump()
    d["cited_section_ids"] = [s for s in dict.fromkeys(d["cited_section_ids"]) if s in state["passages"]]
    return {"draft": d}

RELATION_Q = {"relation": Choice(
    instructions="How does the section relate to the claim?",
    criteria={"supports": "The section states the claim or directly implies that it is true.",
              "contradicts": "The section states the opposite of the claim or implies that it is false.",
              "says_nothing": "The section does not address the claim either way."})}

def verify_c(state: QState) -> dict:
    d = state["draft"]
    if not d["covered"] or not d["cited_section_ids"]:
        return {"grounding": {"claims": []}}
    cited = {s: state["passages"][s] for s in d["cited_section_ids"]}
    extracted = structured(HAIKU, ClaimList, [
        SystemMessage(EXTRACT_PROMPT),
        HumanMessage(f"Answer: {d['answer']}\n\nCited sections:\n\n{format_hits(list(cited.values()))}")])

    def check(c):
        c = c.model_dump()
        section = cited.get(c["cited_section_id"])
        if section is None and len(cited) == 1:            # Haiku left the source blank: only one candidate
            c["cited_section_id"], section = next(iter(cited.items()))
        quote = c["evidence_quote"].strip()
        c["quote_found"] = bool(section and quote and _norm(quote) in _norm(section["text"]))
        c["numbers_ok"] = bool(section) and numbers_match(c["claim"], section["text"])
        if section is None:
            c["verdict"], c["note"] = "unsupported", "cited a section that wasn't retrieved"
            return c
        if quote and not c["quote_found"]:
            c["verdict"], c["note"] = "unsupported", "quote not found in the section"
            return c
        rel = ask_jev({"claim": c["claim"], "section": section["text"]}, RELATION_Q)["relation"]
        c["jev_relation"], c["jev_confidence"] = rel.choice, round(rel.confidence, 3)
        if rel.choice == "contradicts":
            c["verdict"], c["contradicted"], c["note"] = "unsupported", True, f"Jev: contradicts ({rel.confidence:.2f})"
        elif rel.choice == "says_nothing":
            c["verdict"], c["note"] = "unsupported", f"Jev: section says nothing about it ({rel.confidence:.2f})"
        elif not c["numbers_ok"]:
            c["verdict"], c["note"] = "unsupported", "a number in the claim isn't in the section"
        elif not quote:
            c["verdict"], c["note"] = "inferred", "no quote given; Jev judged the whole section"
        elif rel.confidence >= JEV_T["support_conf"]:
            c["verdict"] = "explicit"
        else:
            c["verdict"], c["note"] = "inferred", f"Jev: supports, low confidence ({rel.confidence:.2f})"
        return c

    return {"grounding": {"claims": list(_jev_pool.map(check, extracted.claims))}}

print("Pipeline C nodes defined.")

## 8. Build the graphs and try one question

In [ ]:
from langgraph.graph import StateGraph, START, END

def build(steps):
    b = StateGraph(QState)
    for name, fn in steps:
        b.add_node(name, fn)
    names = [n for n, _ in steps]
    for a, z in zip([START] + names, names + [END]):
        b.add_edge(a, z)
    return b.compile()

GRAPHS = {
    "B": build([("classify", classify_b), ("specialist", specialist_b), ("grounding_check", grounding_b), ("score", score)]),
    "C": build([("jev_classify", classify_c), ("search_and_jev_gate", retrieve_gate_c), ("draft", draft_c),
                ("verify", verify_c), ("score", score)]),
}
graph = GRAPHS[PIPELINE]
try:
    from IPython.display import Image, display
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())

In [ ]:
ICON = {"High": "🟢", "Medium": "🟡", "Low": "🔴"}

def show(state):
    r = state["result"]
    print(f"Q: {r['question']}\nSpecialist: {r['category']}\n\nA: {r['answer']}")
    if r["answer"] == ESCALATE_TEXT and r["draft_answer"]:
        print(f"   (draft for reviewer: {r['draft_answer']})")
    print(f"\nSources: {', '.join(r['sources']) or 'none'}")
    print(f"Confidence: {ICON[r['confidence']]} {r['confidence']}   Status: {r['status']}")
    for why in r["reasons"]:
        print(f"   - {why}")
    print(f"Signals: {r['signals']}")
    for c in state.get("grounding", {}).get("claims", []):
        mark = {"explicit": "✓", "inferred": "~", "unsupported": "✗"}[c["verdict"]]
        print(f"   {mark} [{c['cited_section_id'] or '-'}] {c['claim']}" + (f"   ({c['note']})" if c.get("note") else ""))

def ask(question):
    state = graph.invoke({"question": question})
    show(state)
    return state

# One question costs well under a cent. Comment this out to skip it.
s1 = ask("Within what timeframe will you notify us of a confirmed personal data breach?")
print("\nUsage so far:"); display(cost_table())

## 9. Load the questionnaire and run the pre-flight check
Upload `globex_vendor_due_diligence_2026.xlsx` **and** `golden_set.csv`. Hold Ctrl or Cmd to select both. `RUN_MODE` decides whether all questions run, or just the 10 smoke-test questions.

In [ ]:
from google.colab import files

print("Choose the questionnaire (XLSX or CSV), plus golden_set.csv if you have it.")
try:
    uploaded = files.upload()
except Exception:
    uploaded = {}
candidates = [n for n in uploaded if not n.startswith("golden_set")]
if candidates:
    QFILE = pathlib.Path(candidates[0])
else:
    QFILE = pathlib.Path(sorted(glob.glob("**/data/questionnaires/*.xlsx", recursive=True))[0])
print("Questionnaire:", QFILE)

In [ ]:
import csv, openpyxl

Q_HDR   = re.compile(r"^\s*(question|questions|question text|requirement|control question)\b", re.I)
REF_HDR = re.compile(r"^\s*(ref\.?|reference|q\s*#|q\s*no\.?|#|id|no\.?|item|question id|control id)\s*$", re.I)
ANS_HDRS = [r"supplier response", r"vendor response", r"\banswer\b", r"supporting comments",
            r"\bcomments?\b", r"\bresponse\b"]                    # first match wins
CHOICE_HDR = re.compile(r"^\s*(response|yes\s*/\s*no|y\s*/\s*n|compliant\??)\s*$", re.I)
EVID_HDR = re.compile(r"evidence|supporting doc", re.I)
NOT_OURS = re.compile(r"reviewer|internal|office use|assessor|score|weight", re.I)   # columns we must not fill
SUBQ = re.compile(r"^\s*\(?([a-h]|i{1,3}|iv|v)\)\s+")

def _txt(v):
    return "" if v is None else str(v).strip()

def read_grid(path):
    # -> {sheet: {"rows": {r: [values]}, "merged": {r: text}}}; merged = rows merged across columns
    if path.suffix.lower() == ".csv":
        with open(path, newline="", encoding="utf-8-sig") as f:
            rows = list(csv.reader(f))
        return {"csv": {"rows": {i + 1: r for i, r in enumerate(rows)}, "merged": {}}}
    wb = openpyxl.load_workbook(path)
    out = {}
    for ws in wb.worksheets:
        if ws.sheet_state != "visible":
            continue
        rows = {r: [c.value for c in row] for r, row in enumerate(ws.iter_rows(), start=1)}
        merged = {}
        for rng in ws.merged_cells.ranges:
            if rng.max_col > rng.min_col:
                for r in range(rng.min_row, rng.max_row + 1):
                    merged[r] = _txt(ws.cell(rng.min_row, rng.min_col).value)
        out[ws.title] = {"rows": rows, "merged": merged}
    return out

def find_layout(rows, scan=25):
    for r in sorted(rows)[:scan]:
        hdr = {c: _txt(v) for c, v in enumerate(rows[r]) if _txt(v)}
        q_col = next((c for c, h in hdr.items() if Q_HDR.search(h) and not NOT_OURS.search(h)), None)
        if q_col is None:
            continue
        ans_col = None
        for pat in ANS_HDRS:
            ans_col = next((c for c, h in hdr.items() if c != q_col and re.search(pat, h, re.I)
                            and not NOT_OURS.search(h)), None)
            if ans_col is not None:
                break
        if ans_col is None:
            continue
        ref_col = next((c for c, h in hdr.items() if REF_HDR.match(h)), None)
        choice_col = next((c for c, h in hdr.items() if c not in (ans_col, q_col) and CHOICE_HDR.match(h)), None)
        evid_col = next((c for c, h in hdr.items() if c != ans_col and EVID_HDR.search(h)
                         and not NOT_OURS.search(h)), None)
        return {"header_row": r, "q": q_col, "ref": ref_col, "answer": ans_col,
                "choice": choice_col, "evidence": evid_col, "headers": hdr}
    return None

def extract_questions(path):
    items, report = [], []
    for sheet, g in read_grid(path).items():
        rows, merged = g["rows"], g["merged"]
        lay = find_layout(rows)
        if not lay:
            report.append(f"  {sheet}: no question table found, skipped")
            continue
        h = lay["headers"]
        report.append(f"  {sheet}: header row {lay['header_row']} | question='{h[lay['q']]}' "
                      f"answer='{h[lay['answer']]}'"
                      + (f" ref='{h[lay['ref']]}'" if lay["ref"] is not None else "")
                      + (f" evidence='{h[lay['evidence']]}'" if lay["evidence"] is not None else ""))
        section, n, prefilled = "", 0, 0
        for r in sorted(rows):
            if r <= lay["header_row"]:
                continue
            vals = list(rows[r]) + [None] * 30
            if r in merged:                                   # section header / footer
                section = merged[r]
                continue
            q = _txt(vals[lay["q"]])
            if len(q) < 10 or not re.search(r"[A-Za-z]{3}", q):
                continue
            if _txt(vals[lay["answer"]]):
                prefilled += 1
                continue
            ref = _txt(vals[lay["ref"]]) if lay["ref"] is not None else ""
            model_q = re.sub(r"\s+", " ", q)
            if SUBQ.match(q) and section:                     # a) b) c) -> add parent context
                parent = re.sub(r"^[\d.]+\s*", "", section)
                model_q = f"{parent}: {SUBQ.sub('', model_q)}"
            items.append({"sheet": sheet, "row": r, "ref": ref or f"{sheet}!{r}", "question": q,
                          "model_question": model_q, "section": section,
                          "answer_col": lay["answer"], "evidence_col": lay["evidence"]})
            n += 1
        report[-1] += f"\n      -> {n} questions" + (f", {prefilled} already answered (skipped)" if prefilled else "")
    return items, report

items, report = extract_questions(QFILE)
print(f"{QFILE.name}:")
print("\n".join(report))
print(f"\n{len(items)} questions to answer. First few:")
for it in items[:3] + [i for i in items if i["model_question"] != re.sub(r'\s+', ' ', i["question"])][:2]:
    print(f"  [{it['sheet']} {it['ref']}] {it['model_question'][:110]}")

if RUN_MODE == "smoke":
    items = [i for i in items if i["ref"] in SMOKE_REFS]
print(f"\nRun mode '{RUN_MODE}': {len(items)} questions will be answered by pipeline {PIPELINE}.")

In [ ]:
# ---- Pre-flight: free apart from three tiny API pings ----
checks = []
def check(name, ok, hint=""):
    checks.append(ok)
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  ->  {hint}"))

check(f"Pipeline setting is valid ({PIPELINE})", PIPELINE in GRAPHS, 'set PIPELINE to "B" or "C" in section 1')
check(f"Run mode is valid ({RUN_MODE})", RUN_MODE in ("smoke", "full"), 'set RUN_MODE to "smoke" or "full"')
check("Run 2 fixes present in prompts",
      "True ONLY if" in Classification.model_fields["asks_for_commitment"].description
      and "Do not add related" in DRAFT_PROMPT and 'A bare "Yes."' in GROUNDING_PROMPT, "re-run section 4")
check("Policy index loaded (67 sections)",
      conn.execute("SELECT count(*) FROM passages").fetchone()[0] == 67, "re-run section 2")
expected = len(SMOKE_REFS) if RUN_MODE == "smoke" else 60
check(f"Questions selected: {len(items)} (expected {expected} for the Globex file)",
      len(items) == expected, "check the upload in this section; for another questionnaire this can be ignored")
check("Answer key uploaded (golden_set.csv)",
      bool(glob.glob("**/golden_set*.csv", recursive=True)), "upload golden_set.csv, or section 13 skips scoring")
for model in (HAIKU, SONNET):
    try:
        ChatAnthropic(model=model, max_tokens=5).invoke("Reply with OK")
        check(f"Anthropic key and credits work for {model}", True)
    except Exception as e:
        check(f"Anthropic key and credits work for {model}", False, f"{type(e).__name__}: {str(e)[:150]}")
if PIPELINE == "C":
    try:
        a = ask_jev("The sky is blue.", {"ping": Noul(instructions="The statement is about the sky.")})
        check(f"TypeSafe key works (Jev ping p={a['ping'].noul:.2f})", True)
    except Exception as e:
        check("TypeSafe key works", False, f"{type(e).__name__}: {str(e)[:150]}")

EST_PER_Q = {"B": 0.005, "C": 0.004}
print(f"\nEstimated cost of this run: about ${EST_PER_Q[PIPELINE] * len(items):.2f}")
print("🟢 All clear: run sections 10-13." if all(checks) else "🔴 Fix the ❌ items above before running section 10.")

## 10. Answer the questions in parallel
The same `Send` fan-out as `demo.ipynb`, using the graph for `PIPELINE`. Token counts are reset first, so section 11 shows the cost of this run alone.

In [ ]:
import operator, threading, time
from typing import Annotated
from langgraph.types import Send

MAX_PARALLEL = 6

USAGE.clear()                                # measure this run only
graph = GRAPHS[PIPELINE]

class BatchState(TypedDict):
    items: list
    results: Annotated[list, operator.add]     # each parallel task appends its result

def fan_out(state: BatchState):
    return [Send("answer_one", item) for item in state["items"]]

def answer_one(item: dict) -> dict:
    t0 = time.time()
    try:
        s = graph.invoke({"question": item["model_question"]})
        r = dict(s["result"])
        r["claims"] = s["grounding"]["claims"]
        r["search_log"] = s["search_log"]
        r["source_titles"] = [f"{s['passages'][sid]['doc_title']} §{sid}" for sid in r["sources"]]
        r["pipeline"] = PIPELINE
    except Exception as e:
        r = {"answer": "", "draft_answer": "", "sources": [], "source_titles": [], "category": "",
             "confidence": "Low", "status": "error", "reasons": [f"{type(e).__name__}: {e}"],
             "signals": {}, "claims": [], "search_log": [], "pipeline": PIPELINE}
    r["seconds"] = round(time.time() - t0, 1)
    return {"results": [{**r, **item}]}        # item keys (sheet, ref, original question) win

batch_builder = StateGraph(BatchState)
batch_builder.add_node("answer_one", answer_one)
batch_builder.add_conditional_edges(START, fan_out, ["answer_one"])
batch_builder.add_edge("answer_one", END)
batch_graph = batch_builder.compile()

results, t0 = [], time.time()
print(f"Answering {len(items)} questions, {MAX_PARALLEL} at a time...\n")
for update in batch_graph.stream({"items": items, "results": []},
                                 config={"max_concurrency": MAX_PARALLEL}, stream_mode="updates"):
    for r in update.get("answer_one", {}).get("results", []):
        results.append(r)
        icon = {"auto_approve": "🟢", "human_review": "🟡", "escalate": "🔴", "error": "⚠️"}[r["status"]]
        print(f"{len(results):3}/{len(items)} {icon} {r['status']:12} [{r['sheet'][:18]:18} {r['ref']:7}] {r['question'][:60]}")
order = {(i["sheet"], i["row"]): n for n, i in enumerate(items)}
results.sort(key=lambda r: order[(r["sheet"], r["row"])])
RUN_SECONDS = round(time.time() - t0)
print(f"\nDone in {RUN_SECONDS}s.")

## 11. Summary and measured cost

In [ ]:
df = pd.DataFrame(results)
print(df["status"].value_counts().to_string(), "\n")
print("Auto-answer rate:", f"{(df['status'] == 'auto_approve').mean():.0%}")
errors = df[df["status"] == "error"]
if len(errors):
    print(f"\n⚠️ {len(errors)} errors, e.g.: {errors.iloc[0]['reasons'][0][:200]}")

costs = cost_table()
RUN_COST = costs["cost_usd"].sum()
print(f"\nMeasured cost of this run: ${RUN_COST:.4f}  (${RUN_COST / max(len(df), 1):.4f} per question)")
display(costs.style.format({"cost_usd": "${:.4f}", "input_tokens": "{:,}", "output_tokens": "{:,}"}))

## 12. Save the completed questionnaire, audit trail and cost record
File names include the pipeline and run mode, so runs don't overwrite each other, e.g. `..._C_full_audit.csv`.

In [ ]:
import json
from openpyxl.styles import PatternFill, Font, Alignment

FILL = {"auto_approve": "E2EFDA", "human_review": "FCE4D6", "escalate": "F8CBAD", "error": "D9D9D9"}

def cell_text(r):
    if r["status"] == "auto_approve":
        return r["answer"]
    if r["status"] == "human_review":
        return "[REVIEW] " + r["draft_answer"]
    if r["status"] == "escalate":
        return "[ESCALATE] " + ESCALATE_TEXT
    return "[ERROR] " + "; ".join(r["reasons"])

def claim_counts(r):
    v = [c["verdict"] for c in r["claims"]]
    return f"{v.count('explicit')} explicit / {v.count('inferred')} inferred / {v.count('unsupported')} unsupported"

stem = f"{QFILE.stem}_{PIPELINE}_{RUN_MODE}"
OUT_XLSX, OUT_AUDIT = f"{stem}_COMPLETED.xlsx", f"{stem}_audit.csv"

if QFILE.suffix.lower() == ".xlsx":
    wb = openpyxl.load_workbook(QFILE)
    for r in results:
        ws = wb[r["sheet"]]
        c = ws.cell(r["row"], r["answer_col"] + 1, cell_text(r))
        c.fill = PatternFill("solid", fgColor=FILL[r["status"]])
        c.alignment = Alignment(wrap_text=True, vertical="top")
        if r["evidence_col"] is not None and r["source_titles"]:
            ws.cell(r["row"], r["evidence_col"] + 1, "; ".join(r["source_titles"]))
    if "Responder Summary" in wb.sheetnames:
        del wb["Responder Summary"]
    ws = wb.create_sheet("Responder Summary")
    cols = ["Sheet", "Ref", "Question", "Specialist", "Confidence", "Status", "Answer / draft",
            "Sources", "Reasons", "Claims", "Reviewer", "Reviewer decision"]
    ws.append(cols)
    for r in results:
        ws.append([r["sheet"], r["ref"], r["question"], r["category"], r["confidence"], r["status"],
                   r["answer"] if r["status"] != "human_review" else r["draft_answer"],
                   "; ".join(r["source_titles"]), "; ".join(r["reasons"]), claim_counts(r), "", ""])
        ws.cell(ws.max_row, 6).fill = PatternFill("solid", fgColor=FILL[r["status"]])
    for i, w in enumerate([18, 9, 60, 14, 11, 13, 70, 40, 50, 30, 14, 16]):
        ws.column_dimensions[openpyxl.utils.get_column_letter(i + 1)].width = w
    for row in ws.iter_rows(min_row=1):
        for c in row:
            c.alignment = Alignment(wrap_text=True, vertical="top")
            c.font = Font(name="Arial", size=9, bold=(c.row == 1))
    ws.freeze_panes = "D2"
    ws.auto_filter.ref = ws.dimensions
    wb.save(OUT_XLSX)
else:                                                    # CSV in -> CSV out with answers added
    OUT_XLSX = f"{stem}_COMPLETED.csv"
    rows = list(csv.reader(open(QFILE, newline="", encoding="utf-8-sig")))
    by_row = {r["row"]: r for r in results}
    for i, row in enumerate(rows, start=1):
        if i in by_row:
            r = by_row[i]
            row += [""] * (r["answer_col"] + 1 - len(row))
            row[r["answer_col"]] = cell_text(r)
            row += [r["confidence"], r["status"], "; ".join(r["source_titles"])]
    with open(OUT_XLSX, "w", newline="", encoding="utf-8") as f:
        csv.writer(f).writerows(rows)

audit = pd.DataFrame([{
    "sheet": r["sheet"], "ref": r["ref"], "row": r["row"], "question": r["question"],
    "question_sent_to_model": r["model_question"], "specialist": r["category"],
    "confidence": r["confidence"], "status": r["status"], "answer": r["answer"],
    "draft_answer": r["draft_answer"], "sources": "; ".join(r["sources"]),
    "reasons": " | ".join(r["reasons"]), "signals": json.dumps(r["signals"]),
    "claims": json.dumps(r["claims"]), "searches": " | ".join(r["search_log"]),
    "seconds": r["seconds"], "pipeline": PIPELINE, "reviewer": "", "reviewer_decision": ""} for r in results])
audit.to_csv(OUT_AUDIT, index=False)
OUT_COST = f"{stem}_cost.json"
with open(OUT_COST, "w") as f:
    json.dump({"pipeline": PIPELINE, "run_mode": RUN_MODE, "questions": len(results), "seconds": RUN_SECONDS,
               "cost_usd": round(RUN_COST, 5), "prices": PRICES, "usage": dict(USAGE)}, f, indent=2)
print("Saved:", OUT_XLSX, OUT_AUDIT, OUT_COST)
files.download(OUT_XLSX)
files.download(OUT_AUDIT)
files.download(OUT_COST)

## 13. Check this run against the answer key

In [ ]:
gold_files = sorted(glob.glob("**/golden_set*.csv", recursive=True))
if not gold_files:
    print("No golden_set.csv found; skipping.")
else:
    gold = pd.read_csv(gold_files[0], dtype=str).fillna("")
    gold.loc[gold["ref"] == "3.12", "expected_outcome"] = "medium"   # correction from run 1
    EXPECT = {"high": {"auto_approve"}, "medium": {"human_review"}, "gap": {"escalate"},
              "partial": {"human_review", "escalate"},      # either is safe: a human sees it
              "restricted": {"human_review", "escalate"}}
    m = df.merge(gold, on=["sheet", "ref"], how="inner", suffixes=("", "_gold"))
    m["status_ok"] = [s in EXPECT[e] for s, e in zip(m["status"], m["expected_outcome"])]
    m["routing_ok"] = [c in e.split("|") for c, e in zip(m["category"], m["expected_category"])]
    m["sources_ok"] = [not e or bool(set(e.split(";")) & set(s)) for e, s in zip(m["expected_sources"], m["sources"])]
    print(f"Matched {len(m)} of {len(df)} results to the answer key.\n")
    print(f"Outcome correct:  {m['status_ok'].mean():.0%}")
    print(f"Routing correct:  {m['routing_ok'].mean():.0%}")
    print(f"Cites the expected section (where one exists): {m.loc[m['expected_sources'] != '', 'sources_ok'].mean():.0%}\n")
    print("Expected outcome vs what happened:")
    print(pd.crosstab(m["expected_outcome"], m["status"]).to_string())
    misses = m[~m["status_ok"]][["sheet", "ref", "question", "expected_outcome", "status", "confidence", "reasons"]]
    print(f"\n{len(misses)} mismatches:")
    display(misses)

## 14. Scorecard: A vs B vs C
Builds the comparison from **every audit file in this session**. To include the Opus baseline, upload `run1_audit.csv` from the repo's `results/` folder when asked. Leave the picker empty to skip it.

The baseline's cost is the estimate from run 1, since it was run before cost tracking existed. B and C use the costs measured in section 11.

In [ ]:
import json

if not any("run1" in p for p in glob.glob("*audit*.csv")):
    print("Optional: upload run1_audit.csv (baseline A), or press Cancel to skip.")
    try:
        files.upload()
    except Exception:
        pass

gold = pd.read_csv(sorted(glob.glob("**/golden_set*.csv", recursive=True))[0], dtype=str).fillna("")
gold.loc[gold["ref"] == "3.12", "expected_outcome"] = "medium"
EXPECT = {"high": {"auto_approve"}, "medium": {"human_review"}, "gap": {"escalate"},
          "partial": {"human_review", "escalate"}, "restricted": {"human_review", "escalate"}}

rows = []
for path in sorted(glob.glob("*audit*.csv")):
    a = pd.read_csv(path, dtype=str).fillna("")
    label = a["pipeline"].iloc[0] if "pipeline" in a and a["pipeline"].iloc[0] else "A"
    mode = "smoke" if "_smoke_" in path else "full" if len(a) >= 60 else f"{len(a)} q"
    m = a.merge(gold, on=["sheet", "ref"], suffixes=("", "_gold"))
    ok = [s in EXPECT[e] for s, e in zip(m["status"], m["expected_outcome"])]
    by = lambda outcome: m["expected_outcome"] == outcome
    cost_file = path.replace("_audit.csv", "_cost.json")
    if os.path.exists(cost_file):
        cost = json.load(open(cost_file))["cost_usd"]; cost_note = "measured"
    elif label == "A" and len(a) >= 60:
        cost = 2.20; cost_note = "estimated"
    else:
        cost, cost_note = float("nan"), ""
    rows.append({
        "pipeline": label, "run": mode, "questions": len(m),
        "outcome correct": f"{sum(ok)}/{len(m)}",
        "traps escalated": f"{(m[by('gap')]['status'] == 'escalate').sum()}/{by('gap').sum()}",
        "restricted held": f"{(m[by('restricted')]['status'] != 'auto_approve').sum()}/{by('restricted').sum()}",
        "answerable auto-approved": f"{(m[by('high')]['status'] == 'auto_approve').sum()}/{by('high').sum()}",
        "unsafe auto-approvals": int(((m["status"] == "auto_approve") & ~by("high")).sum()),
        "routing correct": f"{sum(c in e.split('|') for c, e in zip(m['specialist'], m['expected_category'])) / max(len(m), 1):.0%}",
        "cost (USD)": f"${cost:.2f} ({cost_note})" if cost == cost else "-",
        "cost per question": f"${cost / max(len(a), 1):.4f}" if cost == cost else "-",
        "file": path})

board = pd.DataFrame(rows).sort_values(["run", "pipeline"])
display(board)
print("'unsafe auto-approvals' counts anything auto-approved that the answer key says needs a human. It should be 0.")

---
**Reading the scorecard.**
- *Unsafe auto-approvals* must stay at **0**. Any other number disqualifies a pipeline, however cheap.
- After that, compare *answerable auto-approved*: how much work each pipeline saves a human.
- Then compare cost.

Pipeline A ran before the run 2 prompt fixes, so B and C have a head start on over-caution. The run 2 audit shows the fixes alone were worth +7 on the 21 questions re-run.